---
---
# Part 8. [오후] 토큰(Token) — 문장을 조각내고 번호표 붙이기

> **왜 배우나요?** 컴퓨터는 글자를 모름. 트랜스포머에 문장을 넣으려면  
> ① 문장을 조각(토큰)으로 자르고 → ② 각 조각에 번호표(ID)를 붙여야 함  
> 이것이 모든 언어 모델의 입구, **토큰화(tokenization)**

- **토큰(token)** = 문장을 자른 조각. 여기서는 간단하게 "단어 = 토큰"으로 진행  
  (실제 GPT 등은 단어보다 잘게 자르는 subword 방식을 쓰지만 원리는 같음)
- **어휘 사전(vocabulary)** = "토큰 → 번호표(ID)" 대응표. 단어 사전의 번호표라고 생각하면 됨
- **토큰 ID 시퀀스** = 문장을 번호의 나열로 바꾼 것. shape은 `(토큰 개수,)` = `(seq_len,)`

### [질문] 여러 문장으로 어휘 사전을 만들려면 어떻게 해야 할까?

"모든 문장을 단어로 쪼갠다 → 중복을 제거한다 → 각 단어에 0번부터 번호를 붙인다"  
— 이걸 파이썬으로 어떻게 쓸지 잠깐 생각해본 뒤, 같이 구현

### [실습]

In [ ]:
import torch

corpus = [
    "the cat sat on the mat",
    "the dog sat on the rug",
    "a cat and a dog played",
]


In [ ]:
# ① 모든 문장을 단어로 쪼개서 중복 없는 단어 집합 만들기



In [ ]:
# ② 단어 → ID 사전(word_to_id) 만들기: {단어: 번호} 형태



In [ ]:
# ③ 새 문장을 토큰 ID 시퀀스(Tensor)로 바꾸기



> 📌 **체크 포인트**  
> **shape 언어로 정리:** 문장 하나 = `(seq_len,)` 모양의 정수 Tensor (`seq_len` = 토큰 개수)
>
> 그런데 이 번호표에는 문제가 있음. `cat=2`, `dog=4`라고 해서 dog가 cat의 2배로 "개답다"는 뜻도 아니고,  
> 번호가 가깝다고 의미가 비슷한 것도 아님. 번호표는 의미를 담지 못함  
> → 다음 파트에서 번호표를 의미를 담은 벡터로 변환

---

# Part 9. 임베딩(Embedding) — 번호표를 의미 좌표로

> **왜 배우나요?** 트랜스포머 내부의 모든 계산은 벡터끼리의 연산  
> 임베딩 = 토큰 번호표를 의미가 담긴 벡터(좌표)로 바꾸는 것. 인코더에 들어가는 실제 입력이 이것

**임베딩 공간:** 단어를 의미 공간 위의 좌표로 바꾸는 것  
- 공간에서 가까운 단어 = 비슷한 의미 (`cat`과 `dog`는 가깝고, `cat`과 `economy`는 멂)
- 여기서는 50차원 공간을 사용 (좌표 하나가 숫자 50개)

**PyTorch의 임베딩 부품: `nn.Embedding(vocab_size, embedding_dim)`**
- 정체는 `(vocab_size, embedding_dim)` 크기의 표(행렬)
- 토큰 ID가 들어오면 → 그 번호의 행을 꺼내주는 것이 전부 (이 셀은 그냥 실행하며 관찰)

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)   # 결과 재현을 위한 랜덤 시드 고정



In [ ]:
# 토큰 ID 시퀀스를 넣으면 → 각 ID의 행을 꺼내 벡터 시퀀스로



### [질문] 그런데 지금 이 벡터들, 의미가 있을까?

없습니다. `nn.Embedding`은 처음엔 랜덤 숫자로 시작함. 원래는 대량의 텍스트로 학습을 시켜야  
"cat과 dog는 가깝게"라는 의미가 생기는데, 이틀 안에 그걸 학습시킬 수는 없음

**해결책: 이미 학습된 벡터를 가져와서 사용**

word2vec 계열의 사전학습 단어 벡터(GloVe)를 사용  
위키피디아 60억 단어로 미리 학습된, 의미가 이미 담겨 있는 50차원 벡터  
`gensim` 라이브러리로 한 줄이면 불러올 수 있음

In [ ]:
# 사전학습 단어 벡터 로드 (최초 1회만 다운로드(~66MB), 이후엔 캐시에서 바로 로드)
import gensim.downloader as gensim_api

glove = gensim_api.load("glove-wiki-gigaword-50")

In [ ]:
print("로드 완료!")
print("전체 어휘 개수:", len(glove.key_to_index))         
print("벡터 하나의 길이(차원):", glove.___)               
print("\n'cat' 벡터:\n", glove[___])
print("\n'cat' 벡터 차원:\n", len(glove[___]))

In [ ]:
# 의미가 정말 담겨 있을까? — cat과 가장 가까운(비슷한) 단어 Top 5
print("'cat'의 이웃들:")


print("\n'king'의 이웃들:")


# "cat" 대신 좋아하는 영어 단어를 넣어 보세요 ("pizza", "korea", "happy", ...)

### [실습] GloVe 벡터를 우리의 `nn.Embedding`에 이식하기

우리 사전(10단어)에 해당하는 GloVe 벡터만 뽑아서 `nn.Embedding`의 가중치(표)에 복사해 넣습니다.
이러면 **랜덤이 아닌, 의미가 담긴 임베딩 레이어**가 완성됩니다. 모두 같은 값을 쓰게 되니 결과도 똑같아집니다.


In [ ]:
import numpy as np
import torch
import torch.nn as nn

# ① 우리 사전의 단어 순서(ID 순서)대로 GloVe 벡터를 쌓아 행렬 만들기



In [ ]:
# ② Embedding을 만들고, 그 가중치(weight)에 glove_matrix를 복사해 넣기




In [ ]:
# ③ "the cat sat on the mat"의 token_ids를 통과시켜 shape 확인




> 📌 **체크포인트**  
> **shape 흐름 정리 (여기까지 온 과정)**
>
> | 단계 | 데이터 | shape |
> |---|---|---|
> | 문장 | `"the cat sat on the mat"` | (문자열) |
> | 토큰화 | `[7, 2, 8, ...]` | `(seq_len,)` = `(6,)` |
> | **임베딩** | 의미 벡터들 | `(seq_len, embedding_dim)` = `(6, 50)` |
>
> 앞으로 `(seq_len, embedding_dim)` 이 모양이 계속 등장함. 행 = 토큰, 열 = 채널(의미 차원)

---
## 9.4 (확장) 이미지도 토큰화 + 임베딩이 될까? — ViT 맛보기

지금까지는 문장으로만 진행했음. 그런데 트랜스포머는 이미지도 처리함 (**ViT**, Vision Transformer)
비결: **이미지도 "토큰 시퀀스"로 바꾸면 됨**

| | 문장 | 이미지 |
|---|---|---|
| 토큰화 | `split()` → 단어들 | **패치(patch)**: 작은 정사각형 조각으로 자르기 |
| 토큰의 정체 | 번호표(ID) | 픽셀 값 뭉치 |
| 임베딩 방법 | `nn.Embedding` (표에서 행 꺼내기) | `nn.Linear` (픽셀 뭉치 → 벡터 변환) |
| 결과 | `(seq_len, embedding_dim)` | `(패치 개수, embedding_dim)` |

직접 이미지를 잘라서(토큰화) 눈으로 확인해 보자


In [ ]:
# 이미지 로드 (그대로 실행) — matplotlib 대신 PIL 사용 (webp 등 다양한 포맷을 알아서 읽어줌)
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image


# permute(2, 0, 1) = 차원 순서 바꾸기: (높이, 너비, 채널) → (채널, 높이, 너비) ← 파이토치 이미지 관례



In [ ]:
# 이미지의 토큰화 = 패치(patch)로 자르기



# ① 원본 위에 "절취선" 그리기 — 어디가 토큰 경계인지 눈으로 확인



# ② 실제로 잘라서 patches 리스트에 담고, 잘린 조각(토큰)들을 나란히 시각화




In [ ]:
# [실습] 패치(이미지 토큰) 12개를 임베딩 벡터 시퀀스로!
import torch.nn as nn

# ① 패치 리스트를 쌓아 하나의 Tensor로 (힌트: torch.stack) → 예상 shape: (12, 3, 160, 160)


# ② 각 패치를 1차원 벡터로 펴기 (힌트: .flatten(start_dim=1) → 0번째(토큰) 차원은 남기고 나머지만 폄)
#    → 예상 shape: (12, 76800)


# ③ nn.Linear로 각 패치 벡터를 50차원으로 변환 (문장 임베딩과 같은 50차원!) → 예상 shape: (12, 50)



> 📌 **체크포인트 — 문장과 이미지, 결국 같은 모양!**
>
> | | 문장 | 이미지 |
> |---|---|---|
> | 토큰화 | split → 단어 6개 | 패치 자르기 → 조각 12개 |
> | 임베딩 | `nn.Embedding` | `nn.Linear` |
> | 결과 shape | **(6, 50)** | **(12, 50)** |
>
> 둘 다 `(토큰 개수, embedding_dim)` — 그래서 이미지도 **같은 트랜스포머**에 들어갈 수 있음 (ViT의 원리!)
> 오늘 실습은 계속 문장으로 진행함
>
> 🎮 [가지고 놀기] `patch_size`를 80으로 바꾸면 토큰이 몇 개가 될까? (6×8 = 48개!)


## 9.5 (참고) 순서 정보 추가 — Positional Encoding

한 가지 빠진 게 있음. 트랜스포머는 모든 토큰을 동시에 처리해서 순서 정보가 없음  
("고양이가 개를 물었다"와 "개가 고양이를 물었다"를 구분 불가)  
해결책: 위치마다 고유한 벡터를 만들어 임베딩에 더해줌

이 부분은 빈칸 없이 그대로 실행만 진행 (표준 sin/cos 구현 — 암기 불필요, "위치마다 다른 패턴을 더한다"만 기억)

In [ ]:
# unique words를 출력하여 확인
print(unique_words)

In [ ]:
import math
import torch


def create_positional_encoding(max_len, embedding_dim):
    """위치마다 고유한 sin/cos 패턴 벡터를 만들어 (max_len, embedding_dim) 행렬로 반환"""
    position = torch.arange(max_len).unsqueeze(1).float()              # (max_len, 1) 위치 번호 0,1,2,...
    div_term = torch.exp(torch.arange(0, embedding_dim, 2).float()
                         * (-math.log(10000.0) / embedding_dim))       # 주파수를 다양하게
    positional_encoding = torch.zeros(max_len, embedding_dim)
    positional_encoding[:, 0::2] = torch.sin(position * div_term)      # 짝수 채널: sin
    positional_encoding[:, 1::2] = torch.cos(position * div_term)      # 홀수 채널: cos
    return positional_encoding


positional_encoding = create_positional_encoding(max_len=100, embedding_dim=50)
print("positional_encoding shape:", positional_encoding.shape, " ← (max_len, embedding_dim)")

In [ ]:
# 위치 도장이 위치마다 정말 다르게 생겼는지 눈으로 확인
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 4))
plt.imshow(positional_encoding.numpy(), aspect="auto", cmap="RdBu")
plt.xlabel("embedding dimension (channel)")
plt.ylabel("position")  # (몇 번째 토큰인지)
plt.title("Positional Encoding")   # 각 행(위치)이 서로 다른 패턴
plt.colorbar()
plt.show()

In [ ]:
# 사용법은 이게 전부: 문장 길이만큼 잘라서 임베딩에 "더하기"
sequence_length = embedded_sentence.shape[0]     # 6

position_aware_sentence = embedded_sentence + positional_encoding[:sequence_length]
print("임베딩            :", embedded_sentence.shape)
print("위치 도장 (잘라서) :", positional_encoding[:sequence_length].shape)
print("더한 결과          :", position_aware_sentence.shape, " ← shape은 그대로이고 정보만 추가됨")